# Context

In this notebook we will develop some exploratory data analysis

# Load packages

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# from scipy.stats import chi2_contingency
import warnings

warnings.filterwarnings("ignore")
pd.options.display.max_columns = 100
pd.options.display.max_rows = 100

# Load data

using relative paths

In [ ]:
filename = "EDA_regression.ipynb"  # Current file name
print(f"Current file name: {filename}\n")
print(f"Current absolute path: {os.getcwd()}\n")

# Specify the paths, relative to the current file
ACTUAL_DIR = os.path.dirname(os.path.abspath(filename))
BASE_DIR = os.path.dirname(ACTUAL_DIR)
DATA_DIR = os.path.join(BASE_DIR, "data")
OUTPUT_DIR = os.path.join(DATA_DIR, "output_data")

print(f"BASE_DIR: {BASE_DIR}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

In [ ]:
df = pd.read_csv(os.path.join(OUTPUT_DIR, "suertes_for_reg.csv")).iloc[:, 1:]

df.head(5)

In [ ]:
df.info()

In [ ]:
# Estilos
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

# Análisis univariado de datos

### Variable objetivo

In [ ]:
df.tch.plot()
plt.title("Linea de tiempo - tch", size=18)
plt.show()

df.tch.hist()
plt.title("Histograma de frecuencias - tch", size=18)
plt.show()

### Variables categóricas: conteo de frecuencias


In [ ]:
cat_cols = df.select_dtypes(include="object").columns

for col in cat_cols:
    value_counts = df[col].value_counts()
    num_levels = len(value_counts)

    plt.figure(figsize=(8, 5))

    if num_levels < 10:
        sns.barplot(x=value_counts.index, y=value_counts.values)
        plt.xticks(rotation=45)
    else:
        sns.barplot(y=value_counts.index, x=value_counts.values, orient="h")

    plt.title(f"Distribución de: {col}")
    plt.xlabel(col)
    plt.ylabel("Frecuencia")
    plt.show()

### Analisis variables numéricas


In [ ]:
# Resumen estadístico de variables numéricas
print("Resumen estadístico:")
display(df.describe())

In [ ]:
num_cols = df.select_dtypes(include=["float64", "int64"]).columns.tolist()
num_cols.remove("tch")

# Graficar de a 10 variables por bloque
for i in range(0, len(num_cols), 10):
    subset = num_cols[i : i + 10]
    df[subset].hist(bins=30, figsize=(20, 10))
    plt.suptitle(
        f"Distribución de variables numéricas ({i+1} a {i+len(subset)})", fontsize=16
    )
    plt.tight_layout(rect=[0, 0, 1, 0.96])  # para que no se sobreponga el título
    plt.show()

# Analisis multivariado de datos

### Analisis de correlación - columnas numericas

In [ ]:
# ANÁLISIS BIVARIADO

# 4. Correlación entre variables numéricas
num_cols.append("tch")
cor_matrix = df[num_cols].corr()

plt.figure(figsize=(16, 12))
sns.heatmap(cor_matrix, cmap="coolwarm", annot=False)
plt.title("Mapa de calor de correlaciones")
plt.show()

#### TCH vs variables - total

In [ ]:
# Gráfico de barras para el análisis de correlación con la variable objetivo

corr_target = cor_matrix["tch"].drop("tch")

plt.figure(figsize=(10, 8))
corr_target.sort_values().plot(kind="barh", color="purple")
plt.title("Correlación de características con la variable objetivo (tch)")
plt.xlabel("Correlación")
plt.ylabel("Características")
plt.grid(True)
plt.tight_layout()
plt.show()

#### TCH vs variables - por variedad de caña > 2000

In [ ]:
# Gráfico de barras para el análisis de correlación con la variable objetivo

cor_matrix = df[df["variedad_v2"] == "cc_mayor_2000"][num_cols].corr()
corr_target = cor_matrix["tch"].drop("tch")

plt.figure(figsize=(10, 8))
corr_target.sort_values().plot(kind="barh", color="purple")
plt.title(
    "Correlación de características con la variable objetivo (tch) -- cc_mayor_2000"
)
plt.xlabel("Correlación")
plt.ylabel("Características")
plt.grid(True)
plt.tight_layout()
plt.show()

#### TCH vs variables - por variedad de caña < 2000 y otros paises

In [ ]:
# Gráfico de barras para el análisis de correlación con la variable objetivo

cor_matrix = df[df["variedad_v2"] != "cc_mayor_2000"][num_cols].corr()
corr_target = cor_matrix["tch"].drop("tch")

plt.figure(figsize=(10, 8))
corr_target.sort_values().plot(kind="barh", color="purple")
plt.title(
    "Correlación de características con la variable objetivo (tch) -- cc_menor_2000"
)
plt.xlabel("Correlación")
plt.ylabel("Características")
plt.grid(True)
plt.tight_layout()
plt.show()

### TCH vs variables categoricas vs variedad caña

In [ ]:
# 5. Boxplots de variables categóricas vs. TCH (rendimiento)
target = "tch"
cat_to_plot = [
    "num_cosechas",
    "rango_edad_ult_cos",
    "tipo_quema",
    "t_corte",
    "cult_organico",
    "algun_fertilizante",
    "zona",
]

for col in cat_to_plot:
    if col in df.columns:
        plt.figure(figsize=(14, 5))
        sns.boxplot(data=df, x=col, y=target, hue=df["variedad_v2"])
        plt.title(f"{target} vs {col}")
        plt.xticks(rotation=45)
        plt.show()

### TCH vs variables numericas

Solo las variables numericas con mayor valor absoluto de correlación de pearson

In [ ]:
# 6. Scatterplots de algunas variables numéricas con TCH
import math

target = "tch"
num_cols = (
    df.select_dtypes(include=["float64", "int64"]).columns.drop(["tch"]).to_list()
)

columns_to_drop = corr_target[abs(corr_target) < 0.01].index.tolist()

num_cols = list(set(num_cols) - set(columns_to_drop))

# Bloques de 6 gráficos
cols_per_fig = 2
rows_per_fig = 3
plots_per_fig = cols_per_fig * rows_per_fig

for i in range(0, len(num_cols), plots_per_fig):
    subset = num_cols[i : i + plots_per_fig]
    fig, axes = plt.subplots(rows_per_fig, cols_per_fig, figsize=(15, 10))
    axes = axes.flatten()

    for j, col in enumerate(subset):
        corr = corr_target.loc[col]
        sns.scatterplot(
            data=df, x=col, y=target, alpha=0.4, hue="variedad_v2", ax=axes[j]
        )
        axes[j].set_title(f"{target} vs {col} -- Corr: {round(corr,2)}", fontsize=10)

    # Ocultar ejes vacíos (si hay)
    for k in range(j + 1, len(axes)):
        axes[k].axis("off")

    plt.tight_layout()
    plt.suptitle("Scatterplots: Variables numéricas vs. TCH", fontsize=16, y=1.02)
    plt.show()